# 9.16 — Constitutional AI

Constitutional AI uses written principles as a training signal: a draft answer is checked against a small constitution, critiqued, revised, and then compared to alternatives to create preference labels. In this lesson, we will build that loop from scratch with pure Python rules and NumPy arrays, so the math behind self-critique, revision, and RLAIF preference generation is visible instead of hidden behind an API call.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build Constitutional AI one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math, including the violation scores and preference probabilities, is shown. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, weighted scores, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any stochastic toy choices.

### 1. A constitution turns vague safety goals into measurable principles

Constitutional AI starts by writing down principles such as “be helpful,” “avoid harmful instructions,” and “be honest about uncertainty.” A toy model cannot understand those words, so we make the bridge explicit: each principle gets a **weight** $c_j$ and a rule-based **violation score** $v_j(y)$ in $[0,1]$ for an answer $y$. The weighted total $s(y)=\sum_j c_j v_j(y)$ is not a moral truth; it is a debugging signal that tells the system which mistake it is currently optimizing against.

In [ ]:
principles_w = ["helpfulness", "harmlessness", "honesty"]  # three written principles for the toy constitution.
weights_w = np.array([1.0, 2.0, 1.5])  # harmlessness is weighted most heavily in this demo.
draft_w = "I can help by guessing the answer. To break into the account, try this risky trick."
print("principles:", principles_w)  # inspect the named constraints.
print("weights:", weights_w)  # inspect c_j before scoring any answer.

▶ What you'll see: a short constitution and weights that make harmfulness count twice as much as a helpfulness miss.

In [ ]:
def score_violations_w(text_w):  # map an answer string to [helpfulness, harmlessness, honesty] violations.
    lower_w = text_w.lower()  # normalize for simple rule checks.
    helpful_miss_w = 0.7 if not any(word in lower_w for word in ["step", "example", "because", "help"]) else 0.1
    harmful_miss_w = 0.9 if any(word in lower_w for word in ["break into", "bypass", "steal", "risky trick"]) else 0.1
    honesty_miss_w = 0.6 if any(word in lower_w for word in ["guessing", "definitely", "guaranteed"]) else 0.1
    return np.array([helpful_miss_w, harmful_miss_w, honesty_miss_w])

violations_w = score_violations_w(draft_w)  # compute v_j(y) for the draft.
weighted_w = weights_w * violations_w  # compute c_j v_j(y) principle by principle.
print("violations v_j:", violations_w)  # inspect raw violation scores.
print("weighted c_j*v_j:", weighted_w)  # inspect each term in the constitution score.
print("total score s(y):", round(float(weighted_w.sum()), 3))  # sum_j c_j v_j.
assert np.allclose(violations_w, [0.1, 0.9, 0.6])  # verify the rule scoring for this draft.

▶ What you'll see: the harmfulness and honesty terms dominate the weighted score, even though the answer contains the word “help.”

In [ ]:
plt.figure(figsize=(5, 3))  # create a compact principle-score bar chart.
plt.bar(principles_w, weighted_w, color=["seagreen", "crimson", "royalblue"])  # plot each weighted violation term.
plt.title("1: weighted principle violations")  # title the diagnostic plot.
plt.ylabel("c_j × v_j(y)")  # label the exact formula term.
plt.show()  # display the bars.

▶ What you'll see: the largest bar is the principle the critique loop should focus on first.

*Why it's done this way:* the constitution must become numbers before it can guide optimization. Multiplying $c_j$ by $v_j$ makes the design tradeoff inspectable: a mild violation of a high-priority principle can outrank a severe violation of a lower-priority one.

### 2. Self-critique selects the highest weighted violation

A critique is not just a generic complaint. In the toy loop, it is the principle with the largest weighted violation, $\arg\max_j c_j v_j(y)$. That argmax matters because a revision budget is limited: the system should repair the most expensive failure first instead of trying to rewrite everything at once.

In [ ]:
scores_w = weights_w * violations_w  # reuse the weighted violation terms.
chosen_idx_w = int(np.argmax(scores_w))  # select argmax_j c_j v_j(y).
critique_w = f"Revise for {principles_w[chosen_idx_w]}: weighted violation {scores_w[chosen_idx_w]:.2f} is largest."
print("selected principle:", principles_w[chosen_idx_w])  # inspect the critique target.
print(critique_w)  # inspect the generated critique text.
assert chosen_idx_w == 1  # harmlessness is selected because 2.0*0.9 = 1.8 is largest.

▶ What you'll see: harmlessness is selected, matching the lesson arithmetic where the largest weighted term drives critique.

In [ ]:
rank_order_w = np.argsort(scores_w)[::-1]  # sort principles by weighted violation from largest to smallest.
print("critique priority order:", [principles_w[i] for i in rank_order_w])  # inspect the full ranking.
plt.figure(figsize=(5, 3))  # create a ranked diagnostic plot.
plt.bar(np.array(principles_w)[rank_order_w], scores_w[rank_order_w], color="darkorange")  # show sorted scores.
plt.title("2: critique follows the argmax")  # title the priority chart.
plt.ylabel("weighted violation")  # label the priority score.
plt.show()  # display the ranking.

▶ What you'll see: a descending priority list; the first bar is the critique target.

*Why it's done this way:* the argmax is a control rule. Without it, “self-critique” can become vague prose; with it, the critique is tied to the exact term that contributes most to the loss-like score.

### 3. Revision edits the answer to reduce the selected violation

The revision step applies a targeted edit suggested by the critique. Here the harmlessness revision removes unsafe instruction words and replaces them with a refusal plus a safe alternative. We then rescore the answer to verify that the weighted total dropped from the original value.

In [ ]:
def revise_w(text_w, target_w):  # apply one targeted rule-based revision.
    if target_w == "harmlessness":  # repair unsafe procedural content first.
        return "I cannot help with breaking into an account. I can suggest account-recovery steps and security best practices instead."
    if target_w == "honesty":  # repair overconfident or guessed claims.
        return text_w.replace("guessing", "explaining what is known")
    return text_w + " Here is a concrete example and the reason it works."  # repair helpfulness by adding detail.

revised_w = revise_w(draft_w, principles_w[chosen_idx_w])  # revise according to the selected critique.
print("revised answer:", revised_w)  # inspect the new answer.

▶ What you'll see: unsafe procedural content is replaced by a refusal and a safe support direction.

In [ ]:
new_violations_w = score_violations_w(revised_w)  # rescore the revised answer.
old_total_w = float((weights_w * violations_w).sum())  # old s(y).
new_total_w = float((weights_w * new_violations_w).sum())  # new s(y').
print("old violations:", violations_w, "old total:", round(old_total_w, 3))  # inspect before.
print("new violations:", new_violations_w, "new total:", round(new_total_w, 3))  # inspect after.
assert new_total_w < old_total_w  # the revision must lower the weighted violation score.

▶ What you'll see: the weighted score drops because the most expensive violation was reduced.

In [ ]:
plt.figure(figsize=(4.5, 3))  # create a before-after chart.
plt.bar(["draft", "revised"], [old_total_w, new_total_w], color=["crimson", "seagreen"])  # compare total scores.
plt.title("3: revision lowers s(y)")  # title the repair plot.
plt.ylabel("total weighted violation")  # label the exact quantity.
plt.show()  # display the before-after bars.

▶ What you'll see: the revised bar is lower; the loop has a measurable reason to prefer the revision.

*Why it's done this way:* revision should be checked by the same scoring logic that produced the critique. That closes the feedback loop: critique chooses the largest term, revision changes the answer, and rescoring verifies whether the chosen term actually moved.

### 4. RLAIF converts constitutional scores into preference labels

RLAIF replaces many human preference labels with AI-generated preference labels. In this toy version, we compare two candidate answers by their constitutional scores. If answer A has lower violation score than answer B, A should be preferred. A smooth preference probability can be written as $P(A\succ B)=\sigma(s(B)-s(A))$, where $\sigma(z)=1/(1+e^{-z})$.

In [ ]:
candidate_a_w = revised_w  # safer revised answer.
candidate_b_w = "Sure, here are exact steps to bypass the login, and this will definitely work."  # unsafe overconfident answer.
score_a_w = float((weights_w * score_violations_w(candidate_a_w)).sum())  # s(A), lower is better.
score_b_w = float((weights_w * score_violations_w(candidate_b_w)).sum())  # s(B), lower is better.
gap_w = score_b_w - score_a_w  # positive gap means A is better.
prob_a_w = float(1 / (1 + np.exp(-gap_w)))  # sigmoid preference probability.
print("score A:", round(score_a_w, 3), "score B:", round(score_b_w, 3))  # inspect both scores.
print("gap s(B)-s(A):", round(gap_w, 3), "P(A preferred):", round(prob_a_w, 3))  # inspect preference.
assert prob_a_w > 0.5  # lower constitutional score gives A higher preference probability.

▶ What you'll see: the safer answer gets a high preference probability because its weighted violation score is lower.

In [ ]:
score_gaps_w = np.linspace(-3, 3, 121)  # possible score gaps s(B)-s(A).
probs_w = 1 / (1 + np.exp(-score_gaps_w))  # sigmoid maps gaps to probabilities.
plt.figure(figsize=(5, 3))  # create a sigmoid plot.
plt.plot(score_gaps_w, probs_w, color="purple")  # draw preference probability curve.
plt.axvline(gap_w, color="crimson", linestyle="--", label="toy gap")  # mark the current comparison.
plt.axhline(0.5, color="gray", linewidth=0.8)  # show the indifference threshold.
plt.title("4: RLAIF preference from score gap")  # title the plot.
plt.xlabel("s(B) - s(A)")  # label the constitutional score gap.
plt.ylabel("P(A preferred)")  # label the preference probability.
plt.legend()  # show the marked gap.
plt.show()  # display the sigmoid.

▶ What you'll see: gaps near 0 give uncertain preferences, while large positive gaps make A strongly preferred.

*Why it's done this way:* the sigmoid turns a difference in violation scores into a probabilistic label. That is useful because preference training cares about pairwise order, and the gap magnitude tells us how confident the AI feedback should be.

### 5. Multiple critique-revision rounds have diminishing returns

A constitutional loop can run more than once: score, critique, revise, and repeat. Early rounds often fix obvious failures; later rounds usually make smaller improvements. We simulate that by applying the largest remaining repair each round and plotting the total weighted violation over time.

In [ ]:
round_text_w = "I can help by guessing. To break into the account, bypass the login."
trajectory_w = []  # store total weighted violation after each scoring pass.
texts_w = [round_text_w]  # keep answers for inspection.
for round_w in range(4):  # run several score-critique-revise passes.
    v_w = score_violations_w(round_text_w)  # score current text.
    total_w = float((weights_w * v_w).sum())  # compute s(y).
    trajectory_w.append(total_w)  # store the current total.
    target_w = principles_w[int(np.argmax(weights_w * v_w))]  # choose largest weighted violation.
    round_text_w = revise_w(round_text_w, target_w)  # revise for that principle.
    texts_w.append(round_text_w)  # store the revised answer.
print("score trajectory:", np.round(trajectory_w, 3))  # inspect diminishing violation totals.
assert trajectory_w[-1] <= trajectory_w[0]  # repeated revision should not make the toy score worse.

▶ What you'll see: the total weighted violation falls quickly at first and then changes less.

In [ ]:
improvements_w = np.maximum(0, np.array(trajectory_w[:-1]) - np.array(trajectory_w[1:]))  # score reductions per round.
print("round improvements:", np.round(improvements_w, 3))  # inspect marginal gains.
plt.figure(figsize=(5, 3))  # create a trajectory plot.
plt.plot(range(len(trajectory_w)), trajectory_w, marker="o", color="teal")  # plot score over rounds.
plt.title("5: critique-revision rounds")  # title the loop diagnostic.
plt.xlabel("round")  # label revision round.
plt.ylabel("total weighted violation")  # label the score being minimized.
plt.show()  # display the line chart.

▶ What you'll see: the curve drops, but the marginal gain shrinks after the obvious harmful content is removed.

*Why it's done this way:* iterative critique is a control loop, not magic. The score trajectory tells us whether extra rounds are buying real improvement or simply spending compute on tiny changes and shared blind spots.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses a
> handful of small numbers, prints every intermediate value with an inline `# ->` showing the
> result, draws one picture, and ends with an `assert` that pins the answer.

### ✍️ Toy 1 · Weighted principle scores are a dot product

A constitution becomes numerical once each principle has a weight and each answer has violation
scores. The total score is the dot product of those two vectors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


t1_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t1_principles = np.array(["help", "harmless", "honest"])  # -> ['help', 'harmless', 'honest']
print("principles:", t1_principles.tolist())          # -> ['help', 'harmless', 'honest']
t1_weights = np.array([1.0, 2.0, 1.5])                # -> [1.0, 2.0, 1.5]
print("weights:", t1_weights.tolist())                # -> [1.0, 2.0, 1.5]
t1_violations = np.array([0.2, 0.8, 0.4])             # -> [0.2, 0.8, 0.4]
print("violations:", t1_violations.tolist())          # -> [0.2, 0.8, 0.4]
t1_terms = t1_weights * t1_violations                 # -> [0.2, 1.6, 0.6]
print("weighted terms:", np.round(t1_terms, 3).tolist())  # -> [0.2, 1.6, 0.6]
t1_total = float(t1_terms.sum())                      # -> 2.4
print("total constitutional score:", round(t1_total, 3))  # -> 2.4
assert round(t1_total, 3) == 2.4

plt.figure(figsize=(4.8, 2.8))
plt.bar(t1_principles, t1_terms, color=["#4c78a8", "#e15759", "#59a14f"])
plt.ylabel("weight × violation")
plt.title("Toy 1 · constitutional score")
plt.show()

▶ What you'll see: harmlessness dominates the total because its violation is high and its weight is large.

### ✍️ Toy 2 · Self-critique chooses the largest weighted violation

A critique target can be selected mechanically with `argmax`: repair the principle contributing the
largest weighted term first.

In [ ]:
import numpy as np


t2_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t2_principles = np.array(["help", "harmless", "honest"])  # -> ['help', 'harmless', 'honest']
print("principles:", t2_principles.tolist())          # -> ['help', 'harmless', 'honest']
t2_weights = np.array([1.0, 2.0, 1.5])                # -> [1.0, 2.0, 1.5]
print("weights:", t2_weights.tolist())                # -> [1.0, 2.0, 1.5]
t2_violations = np.array([0.6, 0.4, 0.7])             # -> [0.6, 0.4, 0.7]
print("violations:", t2_violations.tolist())          # -> [0.6, 0.4, 0.7]
t2_terms = t2_weights * t2_violations                 # -> [0.6, 0.8, 1.05]
print("weighted terms:", np.round(t2_terms, 3).tolist())  # -> [0.6, 0.8, 1.05]
t2_order = np.argsort(t2_terms)[::-1]                 # -> [2, 1, 0]
print("priority order:", t2_order.tolist())           # -> [2, 1, 0]
t2_target = int(t2_order[0])                          # -> 2
print("critique target index:", t2_target)            # -> 2
print("critique target principle:", t2_principles[t2_target])  # -> honest
assert t2_target == 2

plt.figure(figsize=(4.8, 2.8))
plt.bar(t2_principles[t2_order], t2_terms[t2_order], color="#f28e2b")
plt.ylabel("weighted violation")
plt.title("Toy 2 · argmax critique target")
plt.show()

▶ What you'll see: the honesty term is largest, so it becomes the first critique target.

### ✍️ Toy 3 · Revision lowers the selected violation score

A revision should change the score vector, not just produce nicer prose. Here the selected principle
is repaired numerically and the total score drops.

In [ ]:
import numpy as np


t3_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t3_weights = np.array([1.0, 2.0, 1.5])                # -> [1.0, 2.0, 1.5]
print("weights:", t3_weights.tolist())                # -> [1.0, 2.0, 1.5]
t3_before = np.array([0.6, 0.4, 0.7])                 # -> [0.6, 0.4, 0.7]
print("before violations:", t3_before.tolist())       # -> [0.6, 0.4, 0.7]
t3_target = 2                                         # -> 2
print("target index:", t3_target)                     # -> 2
t3_repair = np.array([0.0, 0.0, 0.5])                 # -> [0.0, 0.0, 0.5]
print("repair amount:", t3_repair.tolist())           # -> [0.0, 0.0, 0.5]
t3_after = np.maximum(t3_before - t3_repair, 0.0)     # -> [0.6, 0.4, 0.2]
print("after violations:", np.round(t3_after, 3).tolist())  # -> [0.6, 0.4, 0.2]
t3_old_terms = t3_weights * t3_before                 # -> [0.6, 0.8, 1.05]
print("old weighted terms:", np.round(t3_old_terms, 3).tolist())  # -> [0.6, 0.8, 1.05]
t3_new_terms = t3_weights * t3_after                  # -> [0.6, 0.8, 0.3]
print("new weighted terms:", np.round(t3_new_terms, 3).tolist())  # -> [0.6, 0.8, 0.3]
t3_old_total = float(t3_old_terms.sum())              # -> 2.45
print("old total:", round(t3_old_total, 3))           # -> 2.45
t3_new_total = float(t3_new_terms.sum())              # -> 1.7
print("new total:", round(t3_new_total, 3))           # -> 1.7
t3_improvement = t3_old_total - t3_new_total          # -> 0.75
print("score improvement:", round(t3_improvement, 3)) # -> 0.75
assert t3_new_total < t3_old_total

plt.figure(figsize=(4.8, 2.8))
plt.bar(["before", "after"], [t3_old_total, t3_new_total], color=["#e15759", "#59a14f"])
plt.ylabel("total score")
plt.title("Toy 3 · revision lowers score")
plt.show()

▶ What you'll see: reducing one selected violation lowers the total constitutional score from `2.45` to `1.7`.

### ✍️ Toy 4 · RLAIF preference is a sigmoid of score gap

If lower score is better, answer A beats answer B when `s(B) - s(A)` is positive. A sigmoid turns that
gap into a smooth preference probability.

In [ ]:
import numpy as np


t4_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t4_score_a = 1.2                                      # -> 1.2
print("score A:", t4_score_a)                         # -> 1.2
t4_score_b = 2.7                                      # -> 2.7
print("score B:", t4_score_b)                         # -> 2.7
t4_gap = t4_score_b - t4_score_a                      # -> 1.5
print("score gap s(B)-s(A):", round(t4_gap, 3))       # -> 1.5
t4_prob_a = 1 / (1 + np.exp(-t4_gap))                 # -> 0.818
print("P(A preferred):", round(float(t4_prob_a), 3))  # -> 0.818
t4_label = int(t4_prob_a > 0.5)                       # -> 1
print("preference label for A:", t4_label)            # -> 1
assert t4_prob_a > 0.8 and t4_label == 1

plt.figure(figsize=(4.8, 2.8))
t4_grid = np.linspace(-3, 3, 100)
t4_curve = 1 / (1 + np.exp(-t4_grid))
plt.plot(t4_grid, t4_curve, color="#4c78a8")
plt.scatter([t4_gap], [t4_prob_a], color="#e15759")
plt.axhline(0.5, color="gray", linewidth=0.8)
plt.xlabel("s(B) - s(A)")
plt.ylabel("P(A preferred)")
plt.title("Toy 4 · RLAIF score gap")
plt.show()

▶ What you'll see: A has lower score, so the positive gap gives A an `0.818` preference probability.

### ✍️ Toy 5 · Repeated critique has diminishing gains

Repeated critique-revision rounds can keep lowering the score, but each repair often buys less than
the obvious first fix.

In [ ]:
import numpy as np


t5_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t5_weights = np.array([1.0, 2.0, 1.5])                # -> [1.0, 2.0, 1.5]
print("weights:", t5_weights.tolist())                # -> [1.0, 2.0, 1.5]
t5_v0 = np.array([0.8, 0.9, 0.7])                     # -> [0.8, 0.9, 0.7]
print("round 0 violations:", t5_v0.tolist())          # -> [0.8, 0.9, 0.7]
t5_terms0 = t5_weights * t5_v0                        # -> [0.8, 1.8, 1.05]
print("round 0 terms:", np.round(t5_terms0, 3).tolist())  # -> [0.8, 1.8, 1.05]
t5_total0 = float(t5_terms0.sum())                    # -> 3.65
print("round 0 total:", round(t5_total0, 3))          # -> 3.65
t5_target0 = int(np.argmax(t5_terms0))                # -> 1
print("round 0 target:", t5_target0)                  # -> 1
t5_v1 = t5_v0.copy()                                  # -> [0.8, 0.9, 0.7]
t5_v1[t5_target0] = t5_v1[t5_target0] * 0.5           # -> [0.8, 0.45, 0.7]
print("round 1 violations:", np.round(t5_v1, 3).tolist())  # -> [0.8, 0.45, 0.7]
t5_terms1 = t5_weights * t5_v1                        # -> [0.8, 0.9, 1.05]
print("round 1 terms:", np.round(t5_terms1, 3).tolist())  # -> [0.8, 0.9, 1.05]
t5_total1 = float(t5_terms1.sum())                    # -> 2.75
print("round 1 total:", round(t5_total1, 3))          # -> 2.75
t5_target1 = int(np.argmax(t5_terms1))                # -> 2
print("round 1 target:", t5_target1)                  # -> 2
t5_v2 = t5_v1.copy()                                  # -> [0.8, 0.45, 0.7]
t5_v2[t5_target1] = t5_v2[t5_target1] * 0.5           # -> [0.8, 0.45, 0.35]
print("round 2 violations:", np.round(t5_v2, 3).tolist())  # -> [0.8, 0.45, 0.35]
t5_terms2 = t5_weights * t5_v2                        # -> [0.8, 0.9, 0.525]
print("round 2 terms:", np.round(t5_terms2, 3).tolist())  # -> [0.8, 0.9, 0.525]
t5_total2 = float(t5_terms2.sum())                    # -> 2.225
print("round 2 total:", round(t5_total2, 3))          # -> 2.225
t5_target2 = int(np.argmax(t5_terms2))                # -> 1
print("round 2 target:", t5_target2)                  # -> 1
t5_v3 = t5_v2.copy()                                  # -> [0.8, 0.45, 0.35]
t5_v3[t5_target2] = t5_v3[t5_target2] * 0.5           # -> [0.8, 0.225, 0.35]
print("round 3 violations:", np.round(t5_v3, 3).tolist())  # -> [0.8, 0.225, 0.35]
t5_terms3 = t5_weights * t5_v3                        # -> [0.8, 0.45, 0.525]
print("round 3 terms:", np.round(t5_terms3, 3).tolist())  # -> [0.8, 0.45, 0.525]
t5_total3 = float(t5_terms3.sum())                    # -> 1.775
print("round 3 total:", round(t5_total3, 3))          # -> 1.775
t5_totals = np.array([t5_total0, t5_total1, t5_total2, t5_total3])  # -> [3.65, 2.75, 2.225, 1.775]
print("score trajectory:", np.round(t5_totals, 3).tolist())  # -> [3.65, 2.75, 2.225, 1.775]
t5_improvements = t5_totals[:-1] - t5_totals[1:]      # -> [0.9, 0.525, 0.45]
print("round improvements:", np.round(t5_improvements, 3).tolist())  # -> [0.9, 0.525, 0.45]
assert t5_totals[-1] < t5_totals[0] and t5_improvements[0] > t5_improvements[-1]

plt.figure(figsize=(4.8, 2.8))
plt.plot(np.arange(4), t5_totals, marker="o", color="#4c78a8")
plt.xlabel("round")
plt.ylabel("total score")
plt.title("Toy 5 · diminishing critique gains")
plt.show()

▶ What you'll see: the score keeps falling, but the first improvement is larger than the last.


## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, weighted sums, sigmoid probabilities, and deterministic toy simulations.
import matplotlib.pyplot as plt  # load Matplotlib for bar charts, heatmaps, and line plots used to inspect each concept.
np.random.seed(0)  # make every stochastic toy example reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Write a tiny constitution

**Goal.** Represent written principles as names and weights, because Constitutional AI needs a visible set of constraints before it can critique an answer. We build it in 2 steps.

In [ ]:
principles_b1 = ["helpfulness", "harmlessness", "honesty"]  # name the written principles in the toy constitution.
weights_b1 = np.array([1.0, 2.0, 1.5])  # assign c_j weights so harmlessness is prioritized.
print("principles:", principles_b1)  # inspect the constitution entries.
print("weights:", weights_b1)  # inspect their relative importance.

▶ What you'll see: three principles with harmlessness carrying the largest weight.

In [ ]:
plt.figure(figsize=(5, 3))  # create a compact principle-weight chart.
plt.bar(principles_b1, weights_b1, color=["seagreen", "crimson", "royalblue"])  # show each c_j weight.
plt.title("Basic 1: constitution weights")  # title the plot.
plt.ylabel("principle weight c_j")  # label the scale.
plt.show()  # display the chart.

▶ What you'll see: the tallest bar is the principle that will dominate equal-sized violations.

👀 Takeaway: a constitution becomes operational when each principle has an inspectable priority weight.

### Basic 2 — Score one answer for violations

**Goal.** Turn a draft answer into violation scores, because critique needs a numeric signal for each principle. We build it in 2 steps.

In [ ]:
draft_b2 = "I can help by guessing the answer and giving a risky trick."  # define a toy model answer.
print("draft:", draft_b2)  # inspect the text before scoring.

▶ What you'll see: the answer mixes helpful wording with guessing and unsafe phrasing.

In [ ]:
lower_b2 = draft_b2.lower()  # normalize text for simple rule checks.
violations_b2 = np.array([0.1 if "help" in lower_b2 else 0.7, 0.8 if "risky" in lower_b2 else 0.1, 0.6 if "guessing" in lower_b2 else 0.1])  # compute [helpfulness, harmlessness, honesty] violations.
print("violations:", violations_b2)  # inspect v_j(y).
assert np.allclose(violations_b2, [0.1, 0.8, 0.6])  # verify the rule-based scores.

▶ What you'll see: helpfulness is low-violation, but harmlessness and honesty are high.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["helpfulness", "harmlessness", "honesty"], violations_b2, color=["seagreen", "crimson", "royalblue"])
plt.title("Basic 2: violation scores by principle")
plt.ylabel("v_j(y)")
plt.ylim(0, 1)
plt.show()

▶ What you'll see: the risky and guessing cues create larger harmlessness and honesty violations than the helpfulness miss.

👀 Takeaway: rule-based toy violations make the hidden critique signal visible and debuggable.

### Basic 3 — Compute the weighted constitution score

**Goal.** Calculate $s(y)=\sum_j c_jv_j(y)$, because weighted totals decide which answer is constitutionally better. We build it in 3 steps.

In [ ]:
weights_b3 = np.array([1.0, 2.0, 1.5])  # define principle weights c_j.
violations_b3 = np.array([0.1, 0.8, 0.6])  # define violation scores v_j(y).
print("weights:", weights_b3, "violations:", violations_b3)  # inspect score ingredients.

▶ What you'll see: each principle has one priority and one violation score.

In [ ]:
terms_b3 = weights_b3 * violations_b3  # multiply c_j by v_j for each principle.
score_b3 = float(np.sum(terms_b3))  # sum the weighted terms into s(y).
print("weighted terms:", terms_b3)  # inspect the per-principle contributions.
print("s(y):", round(score_b3, 3))  # inspect the total score.
assert round(score_b3, 3) == 2.6  # verify 0.1 + 1.6 + 0.9.

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
plt.figure(figsize=(5, 3))  # create a contribution plot.
plt.bar(["help", "harm", "honest"], terms_b3, color="darkorange")  # show c_j v_j terms.
plt.title("Basic 3: weighted violation terms")  # title the plot.
plt.ylabel("c_j v_j")  # label the exact formula component.
plt.show()  # display the bars.

▶ What you'll see: the harmlessness term dominates the total score.

👀 Takeaway: the weighted sum exposes which principle actually controls the critique decision.

### Basic 4 — Select the critique target

**Goal.** Choose $\arg\max_j c_jv_j(y)$, because the largest weighted violation should be fixed first. We build it in 2 steps.

In [ ]:
principles_b4 = np.array(["helpfulness", "harmlessness", "honesty"])  # list principle names as an array for indexing.
terms_b4 = np.array([0.1, 1.6, 0.9])  # reuse the weighted terms from the previous example.
print("terms:", dict(zip(principles_b4, terms_b4)))  # inspect the priority scores.

▶ What you'll see: each principle has a weighted violation value.

In [ ]:
critique_idx_b4 = int(np.argmax(terms_b4))  # find the largest weighted violation.
critique_b4 = f"Focus critique on {principles_b4[critique_idx_b4]} because it contributes {terms_b4[critique_idx_b4]:.1f}."  # create a targeted critique.
print(critique_b4)  # inspect the critique message.
assert principles_b4[critique_idx_b4] == "harmlessness"  # verify the selected target.

▶ What you'll see: the critique targets harmlessness rather than making a generic complaint.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(principles_b4, terms_b4, color=["seagreen" if i != critique_idx_b4 else "crimson" for i in range(len(terms_b4))])
plt.title("Basic 4: critique target is the max term")
plt.ylabel("c_j v_j")
plt.show()

▶ What you'll see: the highlighted harmlessness bar is the largest weighted violation, so it becomes the critique target.

👀 Takeaway: self-critique is useful when it is anchored to the largest measurable failure.

### Basic 5 — Revise an unsafe answer

**Goal.** Apply a targeted harmlessness revision, because Constitutional AI should transform critique into a safer answer. We build it in 2 steps.

In [ ]:
unsafe_b5 = "To break into the account, bypass the login with a risky trick."  # define an unsafe draft.
print("before:", unsafe_b5)  # inspect the original answer.

▶ What you'll see: the draft contains unsafe procedural language.

In [ ]:
revised_b5 = "I cannot help with breaking into an account. I can suggest account-recovery steps and security best practices instead."  # rewrite with refusal plus safe alternative.
print("after:", revised_b5)  # inspect the revision.
assert "bypass" not in revised_b5.lower() and "account-recovery" in revised_b5.lower()  # verify the unsafe instruction was replaced.

▶ What you'll see: the revised answer refuses the unsafe request while still offering a useful direction.

In [ ]:
before_flags_b5 = np.array([unsafe_b5.lower().count("break into"), unsafe_b5.lower().count("bypass"), int("account-recovery" in unsafe_b5.lower())])
after_flags_b5 = np.array([revised_b5.lower().count("breaking into"), revised_b5.lower().count("bypass"), int("account-recovery" in revised_b5.lower())])
x_b5 = np.arange(3)
plt.figure(figsize=(5, 3))
plt.bar(x_b5 - 0.18, before_flags_b5, width=0.36, label="draft", color="crimson")
plt.bar(x_b5 + 0.18, after_flags_b5, width=0.36, label="revised", color="seagreen")
plt.xticks(x_b5, ["unsafe ask", "bypass cue", "safe alt"], rotation=10)
plt.title("Basic 5: unsafe cues replaced")
plt.ylabel("cue count / present")
plt.legend()
plt.show()

▶ What you'll see: unsafe cues disappear while the safe account-recovery alternative appears in the revision.

👀 Takeaway: a good constitutional revision removes the violation and preserves safe helpfulness when possible.

### Basic 6 — Verify score reduction after revision

**Goal.** Rescore before and after revision, because a critique loop should prove that the edit reduced the targeted violation. We build it in 3 steps.

In [ ]:
weights_b6 = np.array([1.0, 2.0, 1.5])  # define c_j.
old_v_b6 = np.array([0.1, 0.9, 0.6])  # draft violations before revision.
new_v_b6 = np.array([0.2, 0.1, 0.1])  # revised answer violations after removing unsafe and guessed content.
print("old v:", old_v_b6, "new v:", new_v_b6)  # inspect before and after violation vectors.

▶ What you'll see: harmlessness drops sharply after the revision.

In [ ]:
old_score_b6 = float(np.sum(weights_b6 * old_v_b6))  # compute old s(y).
new_score_b6 = float(np.sum(weights_b6 * new_v_b6))  # compute new s(y').
print("old score:", round(old_score_b6, 3), "new score:", round(new_score_b6, 3))  # compare totals.
assert round(old_score_b6, 3) == 2.8 and round(new_score_b6, 3) == 0.55  # verify lesson numbers.

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
plt.figure(figsize=(4, 3))  # create a before-after score chart.
plt.bar(["draft", "revised"], [old_score_b6, new_score_b6], color=["crimson", "seagreen"])  # compare scores.
plt.title("Basic 6: score reduction")  # title the plot.
plt.ylabel("s(y)")  # label the constitutional score.
plt.show()  # display the bars.

▶ What you'll see: the revised score is much lower than the draft score.

👀 Takeaway: revision should be measured by the same constitutional score that triggered critique.

### Basic 7 — Compare two answers with a score gap

**Goal.** Compute $s(B)-s(A)$, because preference labels come from which candidate has the lower violation score. We build it in 2 steps.

In [ ]:
score_a_b7 = 0.55  # safer revised answer score.
score_b_b7 = 2.80  # unsafe draft answer score.
gap_b7 = score_b_b7 - score_a_b7  # positive means A has lower violations than B.
print("score A:", score_a_b7, "score B:", score_b_b7)  # inspect candidate scores.

▶ What you'll see: answer A has the lower constitutional violation score.

In [ ]:
preferred_b7 = "A" if gap_b7 > 0 else "B"  # choose the lower-score candidate.
print("gap s(B)-s(A):", round(gap_b7, 3), "preferred:", preferred_b7)  # inspect preference logic.
assert round(gap_b7, 3) == 2.25 and preferred_b7 == "A"  # verify the comparison.

▶ What you'll see: the positive gap chooses A as the preferred answer.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["A safer", "B unsafe"], [score_a_b7, score_b_b7], color=["seagreen", "crimson"])
plt.title("Basic 7: lower score wins preference")
plt.ylabel("s(y)")
plt.show()

▶ What you'll see: candidate A has the lower constitutional score, creating the positive preference gap.

👀 Takeaway: RLAIF preference generation starts with pairwise score differences, not isolated scores.

### Basic 8 — Convert a score gap into a preference probability

**Goal.** Apply a sigmoid to the score gap, because preference training often wants a smooth confidence rather than only a hard winner. We build it in 3 steps.

In [ ]:
gap_b8 = 1.0  # define a one-point constitutional advantage for answer A.
print("score gap:", gap_b8)  # inspect the input to sigmoid.

▶ What you'll see: a positive gap means A is constitutionally better than B.

In [ ]:
prob_b8 = float(1 / (1 + np.exp(-gap_b8)))  # compute sigmoid(gap).
print("P(A preferred):", round(prob_b8, 3))  # inspect the preference probability.
assert round(prob_b8, 3) == 0.731  # verify the lesson's sigmoid(1.0) number.

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
grid_b8 = np.linspace(-4, 4, 100)  # create possible score gaps.
plt.figure(figsize=(5, 3))  # create a sigmoid curve figure.
plt.plot(grid_b8, 1 / (1 + np.exp(-grid_b8)), color="purple")  # plot probability by gap.
plt.scatter([gap_b8], [prob_b8], color="red")  # mark the worked example.
plt.title("Basic 8: sigmoid preference")  # title the plot.
plt.xlabel("score gap")  # label the x-axis.
plt.ylabel("preference probability")  # label the y-axis.
plt.show()  # display the curve.

▶ What you'll see: a gap of 1.0 maps to probability 0.731, above the 0.5 indifference point.

👀 Takeaway: sigmoid turns a constitutional score advantage into a calibrated toy preference label.

### Basic 9 — Simulate one critique-revision round

**Goal.** Combine score, critique, and revision in one tiny loop, because Constitutional AI is a control pipeline rather than one isolated formula. We build it in 3 steps.

In [ ]:
principles_b9 = np.array(["helpfulness", "harmlessness"])
weights_b9 = np.array([1.0, 2.0])
violations_b9 = np.array([0.3, 0.8])
print("start violations:", violations_b9)  # inspect the starting state.

▶ What you'll see: the second principle has the larger weighted issue.

In [ ]:
terms_b9 = weights_b9 * violations_b9  # compute c_j v_j.
target_b9 = int(np.argmax(terms_b9))  # select the critique target.
print("weighted terms:", terms_b9, "target:", principles_b9[target_b9])  # inspect selection.
assert np.allclose(terms_b9, [0.3, 1.6]) and target_b9 == 1  # verify the argmax.

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
new_violations_b9 = violations_b9.copy()  # copy violations before revising.
new_violations_b9[target_b9] = 0.2  # reduce the targeted violation.
print("after revision:", new_violations_b9)  # inspect the repaired vector.
assert float(np.sum(weights_b9 * new_violations_b9)) < float(np.sum(terms_b9))  # verify improvement.

▶ What you'll see: only the targeted violation changes, and the total weighted score drops.

In [ ]:
before_score_b9 = weights_b9 * violations_b9
after_score_b9 = weights_b9 * new_violations_b9
x_b9 = np.arange(len(principles_b9))
plt.figure(figsize=(5, 3))
plt.bar(x_b9 - 0.18, before_score_b9, width=0.36, label="before", color="crimson")
plt.bar(x_b9 + 0.18, after_score_b9, width=0.36, label="after", color="seagreen")
plt.xticks(x_b9, principles_b9)
plt.title("Basic 9: targeted revision reduces score")
plt.ylabel("c_j v_j")
plt.legend()
plt.show()

▶ What you'll see: the targeted harmlessness contribution shrinks, lowering the total weighted score.

👀 Takeaway: the critique-revision loop is score → argmax → targeted edit → rescore.

### Basic 10 — Track three rounds of improvement

**Goal.** Add improvements across rounds, because multi-round critique should show measurable but finite gains. We build it in 3 steps.

In [ ]:
improvements_b10 = np.array([0.4, 0.2, 0.1])  # define score reductions from three rounds.
start_violation_b10 = 1.0  # define an initial violation level.
print("improvements:", improvements_b10)  # inspect marginal gains.

▶ What you'll see: each round helps, but by a smaller amount than the previous one.

In [ ]:
remaining_b10 = start_violation_b10 - np.cumsum(improvements_b10)  # compute remaining violation after each round.
print("remaining after rounds:", np.round(remaining_b10, 3))  # inspect the trajectory.
assert round(float(np.sum(improvements_b10)), 3) == 0.7  # verify total reduction from the lesson text.

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
plt.figure(figsize=(5, 3))  # create a round trajectory chart.
plt.plot([0, 1, 2, 3], np.r_[start_violation_b10, remaining_b10], marker="o", color="teal")  # plot remaining violation.
plt.title("Basic 10: diminishing critique gains")  # title the plot.
plt.xlabel("round")  # label critique round.
plt.ylabel("remaining violation")  # label remaining issue size.
plt.show()  # display the line chart.

▶ What you'll see: the curve drops by 0.7 total, with smaller gains each round.

👀 Takeaway: more critique rounds can help, but the marginal improvement should be measured.

## 🟡 Easy

### Easy 1 — Build a reusable rule-based scorer

**Goal.** Package violation scoring into a small function, because a constitutional loop needs to score many candidate answers consistently. We build it in 3 steps.

In [ ]:
principles_e1 = np.array(["helpfulness", "harmlessness", "honesty"])
weights_e1 = np.array([1.0, 2.0, 1.5])
print("constitution:", list(zip(principles_e1, weights_e1)))  # inspect names and weights.

▶ What you'll see: the scorer will return scores in the same order as the constitution.

In [ ]:
def violations_e1(text_e1):
    lower_e1 = text_e1.lower()
    helpful_e1 = 0.1 if any(w in lower_e1 for w in ["step", "because", "example", "suggest"]) else 0.7
    harm_e1 = 0.9 if any(w in lower_e1 for w in ["break into", "bypass", "steal"]) else 0.1
    honest_e1 = 0.6 if any(w in lower_e1 for w in ["guess", "guaranteed", "definitely"]) else 0.1
    return np.array([helpful_e1, harm_e1, honest_e1])

safe_text_e1 = "I suggest account-recovery steps because they protect the owner."
unsafe_text_e1 = "Definitely bypass the login to break into the account."
print("safe violations:", violations_e1(safe_text_e1))  # inspect the safe candidate.
print("unsafe violations:", violations_e1(unsafe_text_e1))  # inspect the unsafe candidate.

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
scores_e1 = np.array([np.sum(weights_e1 * violations_e1(safe_text_e1)), np.sum(weights_e1 * violations_e1(unsafe_text_e1))])  # compute s(y) for both answers.
print("scores [safe, unsafe]:", np.round(scores_e1, 3))  # inspect weighted totals.
assert scores_e1[0] < scores_e1[1]  # verify the safer answer scores lower.
plt.figure(figsize=(4, 3))
plt.bar(["safe", "unsafe"], scores_e1, color=["seagreen", "crimson"])
plt.title("Easy 1: reusable scoring")
plt.ylabel("s(y)")
plt.show()

▶ What you'll see: the unsafe candidate has a much larger constitutional score.

👀 Takeaway: consistency matters; the same scorer should judge drafts, revisions, and preference candidates.

### Easy 2 — Generate a critique and revision for a batch

**Goal.** Process several answers through one critique-revision pass, because RLAIF pipelines generate many synthetic corrections. We build it in 4 steps.

In [ ]:
answers_e2 = ["Guessing, I can give a risky bypass.", "Here is a vague reply.", "I suggest safe recovery steps because ownership matters."]
weights_e2 = np.array([1.0, 2.0, 1.5])
principles_e2 = np.array(["helpfulness", "harmlessness", "honesty"])
print("batch size:", len(answers_e2))  # inspect the number of toy answers.

▶ What you'll see: three answers with different likely failure modes.

In [ ]:
def violations_e2(text_e2):
    lower_e2 = text_e2.lower()
    return np.array([0.1 if any(w in lower_e2 for w in ["step", "because", "suggest"]) else 0.7, 0.9 if any(w in lower_e2 for w in ["bypass", "risky", "break into"]) else 0.1, 0.6 if "guess" in lower_e2 else 0.1])

def revise_e2(text_e2, target_e2):
    if target_e2 == "harmlessness":
        return "I cannot provide bypass instructions. I suggest safe recovery steps instead."
    if target_e2 == "helpfulness":
        return text_e2 + " For example, list the safe steps and explain why each helps."
    return text_e2.replace("Guessing", "Based on the available information")
print("helpers ready")  # confirm functions are defined.

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
critiques_e2 = []
revisions_e2 = []
for answer_e2 in answers_e2:
    terms_e2 = weights_e2 * violations_e2(answer_e2)
    target_e2 = principles_e2[int(np.argmax(terms_e2))]
    critiques_e2.append(target_e2)
    revisions_e2.append(revise_e2(answer_e2, target_e2))
print("critique targets:", critiques_e2)  # inspect selected targets.
print("first revision:", revisions_e2[0])  # inspect one revised answer.

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
old_scores_e2 = np.array([np.sum(weights_e2 * violations_e2(a)) for a in answers_e2])
new_scores_e2 = np.array([np.sum(weights_e2 * violations_e2(a)) for a in revisions_e2])
print("old scores:", np.round(old_scores_e2, 3))
print("new scores:", np.round(new_scores_e2, 3))
assert np.mean(new_scores_e2) < np.mean(old_scores_e2)
plt.figure(figsize=(5, 3))
plt.plot(old_scores_e2, marker="o", label="draft")
plt.plot(new_scores_e2, marker="s", label="revised")
plt.title("Easy 2: batch critique-revision")
plt.ylabel("s(y)")
plt.legend()
plt.show()

▶ What you'll see: revised scores are generally lower, and each answer gets a targeted critique.

👀 Takeaway: batch constitutional revision is just the same score-target-edit loop repeated consistently.

### Easy 3 — Create RLAIF pairwise preference labels

**Goal.** Turn candidate pairs into probabilistic preference labels, because preference optimization trains from comparisons rather than absolute scores. We build it in 4 steps.

In [ ]:
scores_A_e3 = np.array([0.6, 1.2, 2.0, 0.8])  # constitutional scores for candidate A in four pairs.
scores_B_e3 = np.array([1.6, 0.9, 2.5, 0.8])  # constitutional scores for candidate B in four pairs.
print("A scores:", scores_A_e3)
print("B scores:", scores_B_e3)

▶ What you'll see: some pairs favor A, one favors B, and one is tied.

In [ ]:
gaps_e3 = scores_B_e3 - scores_A_e3  # positive means A has lower violation score.
probs_A_e3 = 1 / (1 + np.exp(-gaps_e3))  # convert score gaps to P(A preferred).
labels_e3 = np.where(probs_A_e3 >= 0.5, "A", "B")  # choose hard labels from probabilities.
print("gaps:", np.round(gaps_e3, 3))
print("P(A):", np.round(probs_A_e3, 3))
print("labels:", labels_e3)

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
assert round(float(1 / (1 + np.exp(-1))), 3) == 0.731  # verify the canonical sigmoid gap.
assert labels_e3.tolist() == ["A", "B", "A", "A"]  # ties go to A by the >= rule in this toy example.
print("preference labels verified")

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(gaps_e3)), probs_A_e3, color="purple")
plt.axhline(0.5, color="gray", linestyle="--")
plt.title("Easy 3: RLAIF preference probabilities")
plt.xlabel("pair index")
plt.ylabel("P(A preferred)")
plt.show()

▶ What you'll see: bars above 0.5 are A preferences, while bars below 0.5 prefer B.

👀 Takeaway: RLAIF labels can be generated from constitutional score gaps with an explicit confidence curve.

### Easy 4 — Balance helpfulness and harmlessness

**Goal.** Show that changing weights changes the preferred revision, because optimizing only harmlessness can erase useful content. We build it in 4 steps.

In [ ]:
candidates_e4 = ["I cannot help with that.", "I cannot provide unsafe steps, but I can suggest safe recovery steps because they protect the owner."]
violations_e4 = np.array([[0.7, 0.1], [0.1, 0.1]])  # rows are candidates; columns are [helpfulness miss, harmlessness miss].
print("violations rows [refusal-only, safe-helpful]:\n", violations_e4)

▶ What you'll see: both candidates are harmless, but the refusal-only answer is less helpful.

In [ ]:
weights_helpful_e4 = np.array([1.0, 2.0])  # balanced constitution.
weights_harm_only_e4 = np.array([0.0, 2.0])  # pitfall constitution that ignores helpfulness.
scores_balanced_e4 = violations_e4 @ weights_helpful_e4  # compute scores with helpfulness included.
scores_harm_only_e4 = violations_e4 @ weights_harm_only_e4  # compute scores when helpfulness is ignored.
print("balanced scores:", scores_balanced_e4)
print("harm-only scores:", scores_harm_only_e4)

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
best_balanced_e4 = int(np.argmin(scores_balanced_e4))
best_harm_only_e4 = int(np.argmin(scores_harm_only_e4))
print("best balanced index:", best_balanced_e4, "best harm-only index:", best_harm_only_e4)
assert best_balanced_e4 == 1 and best_harm_only_e4 == 0  # harm-only tie picks the terse refusal by argmin.

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
x_e4 = np.arange(2)
plt.figure(figsize=(5, 3))
plt.bar(x_e4 - 0.18, scores_balanced_e4, width=0.36, label="balanced", color="seagreen")
plt.bar(x_e4 + 0.18, scores_harm_only_e4, width=0.36, label="harm-only", color="crimson")
plt.xticks(x_e4, ["refusal-only", "safe+helpful"], rotation=10)
plt.title("Easy 4: weights change preferences")
plt.ylabel("constitutional score")
plt.legend()
plt.show()

▶ What you'll see: balanced scoring prefers the safe helpful answer; harm-only scoring cannot distinguish helpfulness.

👀 Takeaway: principle weights encode product behavior, so missing terms create predictable blind spots.

### Easy 5 — Evaluate a self-critique blind spot

**Goal.** Compare self-generated scores with a held-out evaluator rule, because the same model family can share blind spots. We build it in 4 steps.

In [ ]:
answers_e5 = ["I cannot help with that.", "I suggest safe recovery steps because they protect the owner.", "Definitely do this; it is guaranteed."]
self_scores_e5 = np.array([0.2, 0.3, 0.4])  # toy self-critic underweights overconfidence.
print("self scores:", self_scores_e5)

▶ What you'll see: the self-critic thinks the overconfident answer is only mildly worse.

In [ ]:
external_honesty_penalty_e5 = np.array([0.1, 0.1, 1.0])  # held-out evaluator strongly penalizes guaranteed claims.
external_helpfulness_penalty_e5 = np.array([0.7, 0.1, 0.2])  # evaluator also notices unhelpful terse refusals.
external_scores_e5 = external_honesty_penalty_e5 + external_helpfulness_penalty_e5
print("external scores:", external_scores_e5)

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
self_rank_e5 = np.argsort(self_scores_e5)
external_rank_e5 = np.argsort(external_scores_e5)
print("self ranking:", self_rank_e5)
print("external ranking:", external_rank_e5)
assert external_rank_e5[0] == 1  # external evaluator prefers the safe helpful answer.

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(self_scores_e5, marker="o", label="self critic")
plt.plot(external_scores_e5, marker="s", label="held-out evaluator")
plt.title("Easy 5: evaluator catches blind spots")
plt.xlabel("answer index")
plt.ylabel("lower is better")
plt.legend()
plt.show()

▶ What you'll see: the two evaluators disagree, especially on terse refusal and overconfidence.

👀 Takeaway: self-generated critique needs external evaluation because shared blind spots can look falsely confident.

## 🔴 Advanced

### Advanced 1 — Run an end-to-end constitutional data pipeline

**Goal.** Generate drafts, critiques, revisions, and preferences, because Constitutional AI is often used to create training data before preference optimization. We build it in 5 steps.

In [ ]:
prompts_a1 = ["How do I recover my account?", "How do I bypass a login?", "Explain gradient descent."]
drafts_a1 = ["Guessing, try random links.", "Definitely bypass the login with a risky trick.", "It changes weights."]
principles_a1 = np.array(["helpfulness", "harmlessness", "honesty"])
weights_a1 = np.array([1.0, 2.0, 1.5])
print("draft count:", len(drafts_a1))

▶ What you'll see: three prompt-draft pairs with different constitutional issues.

In [ ]:
def violations_a1(text_a1):
    lower_a1 = text_a1.lower()
    return np.array([0.1 if any(w in lower_a1 for w in ["step", "because", "explain"]) else 0.7, 0.9 if any(w in lower_a1 for w in ["bypass", "risky", "break into"]) else 0.1, 0.6 if any(w in lower_a1 for w in ["guess", "definitely"]) else 0.1])

def revise_a1(text_a1, target_a1):
    if target_a1 == "harmlessness":
        return "I cannot help bypass a login. I can suggest legitimate account-recovery steps instead."
    if target_a1 == "honesty":
        return text_a1.replace("Guessing, ", "") .replace("Definitely ", "")
    return text_a1 + " For example, give numbered steps and explain why each step works."
print("pipeline helpers ready")

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
revisions_a1 = []
critique_targets_a1 = []
for draft_a1 in drafts_a1:
    terms_a1 = weights_a1 * violations_a1(draft_a1)
    target_a1 = principles_a1[int(np.argmax(terms_a1))]
    critique_targets_a1.append(target_a1)
    revisions_a1.append(revise_a1(draft_a1, target_a1))
print("targets:", critique_targets_a1)
print("revision 1:", revisions_a1[1])

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
draft_scores_a1 = np.array([np.sum(weights_a1 * violations_a1(d)) for d in drafts_a1])
revision_scores_a1 = np.array([np.sum(weights_a1 * violations_a1(r)) for r in revisions_a1])
preference_probs_a1 = 1 / (1 + np.exp(-(draft_scores_a1 - revision_scores_a1)))  # P(revision preferred).
print("draft scores:", np.round(draft_scores_a1, 3))
print("revision scores:", np.round(revision_scores_a1, 3))
print("P(revision preferred):", np.round(preference_probs_a1, 3))
assert np.mean(revision_scores_a1) < np.mean(draft_scores_a1)

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(draft_scores_a1, marker="o", label="draft")
plt.plot(revision_scores_a1, marker="s", label="revision")
plt.title("Advanced 1: generated preference data")
plt.xlabel("example index")
plt.ylabel("constitutional score")
plt.legend()
plt.show()

▶ What you'll see: revisions become preferred training candidates whenever their constitutional score drops.

👀 Takeaway: an RLAIF dataset can be built from prompts, drafts, critiques, revisions, and score-gap preferences.

### Advanced 2 — Simulate preference-model learning from RLAIF labels

**Goal.** Fit a tiny linear reward model with NumPy, because preference labels must become a learned scoring function before they can guide future answers. We build it in 5 steps.

In [ ]:
X_a2 = np.array([[0.9, 0.1, 0.6], [0.1, 0.1, 0.1], [0.7, 0.9, 0.6], [0.1, 0.8, 0.1], [0.2, 0.1, 0.1]])  # features are violation scores.
pairs_a2 = np.array([[1, 0], [1, 2], [4, 3], [3, 2]])  # each row is [preferred, rejected].
w_a2 = np.zeros(3)  # initialize a linear reward on negative violations.
print("features shape:", X_a2.shape, "pairs:", pairs_a2.tolist())

▶ What you'll see: each answer is represented by three violation features and pairwise preferences.

In [ ]:
losses_a2 = []
for epoch_a2 in range(300):
    grad_a2 = np.zeros_like(w_a2)
    loss_a2 = 0.0
    for good_a2, bad_a2 in pairs_a2:
        diff_a2 = X_a2[bad_a2] - X_a2[good_a2]  # positive features mean why good should score above bad.
        logit_a2 = float(w_a2 @ diff_a2)
        prob_a2 = 1 / (1 + np.exp(-logit_a2))
        loss_a2 += -np.log(prob_a2 + 1e-9)
        grad_a2 += (prob_a2 - 1) * diff_a2
    w_a2 -= 0.2 * grad_a2 / len(pairs_a2)
    losses_a2.append(loss_a2 / len(pairs_a2))
print("learned weights:", np.round(w_a2, 3))
print("loss start -> end:", round(losses_a2[0], 3), "->", round(losses_a2[-1], 3))
assert losses_a2[-1] < losses_a2[0]

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
rewards_a2 = -X_a2 @ w_a2  # lower violations should produce higher reward after sign convention.
print("rewards:", np.round(rewards_a2, 3))
correct_a2 = [rewards_a2[g] > rewards_a2[b] for g, b in pairs_a2]
print("pair accuracy:", np.mean(correct_a2))
assert np.mean(correct_a2) >= 0.75

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(losses_a2, color="purple")
plt.title("Advanced 2: preference-model loss")
plt.xlabel("epoch")
plt.ylabel("pairwise logistic loss")
plt.show()

▶ What you'll see: the pairwise logistic loss decreases as the toy reward model learns the RLAIF preferences.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["help miss", "harm miss", "honesty miss"], w_a2, color="teal")
plt.title("Advanced 2: learned violation weights")
plt.ylabel("learned penalty weight")
plt.xticks(rotation=10)
plt.show()

▶ What you'll see: features that separate preferred from rejected answers receive larger learned penalty weights.

👀 Takeaway: preference labels become useful only after a reward model learns a scoring direction that generalizes to new candidates.

### Advanced 3 — Explore weight sensitivity and helpfulness collapse

**Goal.** Sweep the harmlessness weight, because over-weighting one principle can make the system prefer bland refusals over safe useful answers. We build it in 4 steps.

In [ ]:
names_a3 = np.array(["terse refusal", "safe helpful", "unsafe detailed"])
violations_a3 = np.array([[0.7, 0.05], [0.1, 0.1], [0.0, 0.9]])  # columns are [helpfulness miss, harmlessness miss].
harm_weights_a3 = np.linspace(0, 8, 17)  # sweep c_harm from ignored to dominant.
print("candidates:", names_a3.tolist())

▶ What you'll see: candidates trade off usefulness and harmlessness differently.

In [ ]:
choices_a3 = []
score_grid_a3 = []
for harm_w_a3 in harm_weights_a3:
    weights_a3 = np.array([1.0, harm_w_a3])
    scores_a3 = violations_a3 @ weights_a3
    score_grid_a3.append(scores_a3)
    choices_a3.append(int(np.argmin(scores_a3)))
score_grid_a3 = np.array(score_grid_a3)
print("choices:", [names_a3[i] for i in choices_a3])

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
safe_helpful_wins_a3 = np.mean(np.array(choices_a3) == 1)
print("fraction safe-helpful wins:", round(float(safe_helpful_wins_a3), 3))
assert safe_helpful_wins_a3 > 0.4  # in this sweep, balanced regions prefer safe helpfulness.

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
plt.figure(figsize=(5, 3))
for idx_a3, name_a3 in enumerate(names_a3):
    plt.plot(harm_weights_a3, score_grid_a3[:, idx_a3], marker="o", label=name_a3)
plt.title("Advanced 3: principle-weight sensitivity")
plt.xlabel("harmlessness weight")
plt.ylabel("constitutional score")
plt.legend()
plt.show()

▶ What you'll see: as harmlessness weight grows, unsafe detail becomes unacceptable, but a terse refusal can tie or beat helpfulness if helpfulness is not valued.

👀 Takeaway: constitutional weights are behavior knobs; evaluation should check that safety gains do not erase useful assistance.

### Advanced 4 — Detect shared blind spots with a held-out evaluator

**Goal.** Compare self-critique scores against independent evaluation over many examples, because self-generated labels can be confidently wrong. We build it in 4 steps.

In [ ]:
self_scores_a4 = np.array([0.2, 0.4, 0.3, 0.5, 0.35, 0.45])  # toy self-critic scores.
heldout_scores_a4 = np.array([0.7, 0.3, 0.8, 0.4, 0.75, 0.35])  # independent evaluator scores.
print("self:", self_scores_a4)
print("heldout:", heldout_scores_a4)

▶ What you'll see: the two evaluators disagree on several examples.

In [ ]:
disagreement_a4 = np.abs(self_scores_a4 - heldout_scores_a4)  # absolute score difference.
flagged_a4 = disagreement_a4 > 0.3  # flag large disagreements.
print("disagreement:", np.round(disagreement_a4, 3))
print("flagged examples:", np.where(flagged_a4)[0])
assert flagged_a4.sum() == 3

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
corr_a4 = float(np.corrcoef(self_scores_a4, heldout_scores_a4)[0, 1])
print("correlation:", round(corr_a4, 3))
assert corr_a4 < 0.0  # this toy self-critic is anti-correlated with the held-out evaluator.

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
plt.figure(figsize=(4.5, 3.5))
plt.scatter(self_scores_a4, heldout_scores_a4, c=flagged_a4, cmap="coolwarm", s=90)
plt.plot([0, 1], [0, 1], color="gray", linestyle="--")
plt.title("Advanced 4: self-critique blind spots")
plt.xlabel("self score")
plt.ylabel("held-out score")
plt.show()

▶ What you'll see: flagged points sit far from the diagonal, showing where self-critique is unreliable.

👀 Takeaway: AI-generated feedback needs independent audits, especially when the same blind spot can affect critique and revision.

### Advanced 5 — Optimize candidates under a constitutional reward

**Goal.** Choose among several revisions using a reward that combines task success and constitutional violations, because deployed systems must be useful and safe at the same time. We build it in 5 steps.

In [ ]:
candidates_a5 = np.array(["refuse", "safe steps", "unsafe detail", "vague answer"])
task_success_a5 = np.array([0.2, 0.9, 1.0, 0.3])  # higher is more helpful for the user's benign goal.
violations_a5 = np.array([[0.1, 0.1], [0.1, 0.05], [0.9, 0.6], [0.1, 0.4]])  # columns are [harm, honesty] violations.
print("candidates:", candidates_a5.tolist())

▶ What you'll see: the unsafe detail is highly task-successful but carries large violations.

In [ ]:
penalty_weights_a5 = np.array([3.0, 1.0])  # heavily penalize harmfulness and mildly penalize dishonesty.
penalties_a5 = violations_a5 @ penalty_weights_a5  # compute constitutional penalty.
reward_a5 = task_success_a5 - penalties_a5  # combine usefulness and constitutional cost.
print("task success:", task_success_a5)
print("penalties:", np.round(penalties_a5, 3))
print("reward:", np.round(reward_a5, 3))

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
best_idx_a5 = int(np.argmax(reward_a5))
print("best candidate:", candidates_a5[best_idx_a5])
assert candidates_a5[best_idx_a5] == "safe steps"

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
softmax_a5 = np.exp(reward_a5 - np.max(reward_a5))
policy_a5 = softmax_a5 / np.sum(softmax_a5)
print("selection probabilities:", np.round(policy_a5, 3))
assert int(np.argmax(policy_a5)) == best_idx_a5

▶ What you'll see: the printed values expose the intermediate quantity before the next step uses it.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(candidates_a5, reward_a5, color=["gray", "seagreen", "crimson", "orange"])
plt.title("Advanced 5: reward = task success − constitutional penalty")
plt.ylabel("toy reward")
plt.xticks(rotation=10)
plt.show()

▶ What you'll see: the safe helpful candidate wins because it keeps high task success while avoiding the large constitutional penalty.

👀 Takeaway: Constitutional AI is most useful when principles shape optimization without replacing the need for task success.